In [119]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import os
from scrape_wiki import scrape_wiki_table
from get_gender_data import get_founder_gender
from filter_df import filter_df

# Ensure output directory exists
os.makedirs("processed_data", exist_ok=True)

In [120]:
url = "https://en.wikipedia.org/wiki/List_of_unicorn_startup_companies"
idx = 2  # Changed to target the main list of unicorn companies

currentUnicorns_df = scrape_wiki_table(url, idx, "current_unicorns.csv")
currentUnicorns_df.to_csv("current_unicorns.csv")

pastUnicorns_df = scrape_wiki_table(url, idx+1, "past_unicorns.csv")
pastUnicorns_df.to_csv("past_unicorns.csv")

Found 4 wikitable(s)
Saved 618 rows to current_unicorns.csv
Found 4 wikitable(s)
Saved 206 rows to past_unicorns.csv


In [121]:
currentUnicorns_df = pd.read_csv("current_unicorns.csv", index_col=0)
pastUnicorns_df = pd.read_csv("past_unicorns.csv", index_col=0)



In [122]:
display(currentUnicorns_df)
display(pastUnicorns_df)

,Company,Valuation(US$ billions),Valuation date,Industry,Country/countries,Founder(s)
0,SpaceX,1250,February 2026(2026-02)[20],"Aerospace,Artificial Intelligence",United States,Elon Musk
1,OpenAI,852,March 2026(2026-03)[21],Artificial intelligence,United States,"Sam Altman,Elon Musk,Greg Brockman,Ilya Sutskever"
2,Anthropic,380,February 2026(2026-02)[22],Artificial Intelligence,United States,"Dario Amodei,Daniela Amodei,Jared Kaplan, Jack..."
3,ByteDance,600,April 2026[23],Internet,China,"Zhang Yiming, Liang Rubo"
4,Stripe,159,February 2026(2026-02)[24],Financial services,United States and Ireland,PatrickandJohn Collison
...,...,...,...,...,...,...
613,HMD Global,1+,August 2020[571],Mobile Devices,Finland,Jean-Francois Baril
614,IQM,1,July 2022[572],Quantum Computing,Finland,"Jan Goetz, Mikko Möttönen, Kuan Yen Tan, Juha ..."
615,Hostaway,1,December 2024[573],Vacation rental software platform,Finland,"Marcus Räder, Saber Kordestanchi, Mikko Nurminen"
616,Papara,1,July 2023[574],Fintech,Turkey,Ahmed Faruk Karslı


,Company,Last valuation(US$billions),Valuation date,Exit date,Exit reason,Exit valuation(US$billions),Country,Founders,col_8
0,Uber,72,August 2018[577],May 2019[578],IPO,82.4,United States,"Travis Kalanick,Garett Camp",NaN
1,DiDi,62,July 2019[579],June 2021[580],IPO,73,China,Cheng Wei,NaN
2,Facebook,50,January 2011,May 2012[581],IPO,104,United States,"Mark Zuckerberg,Eduardo Saverin, Andrew McColl...",NaN
3,Xiaomi,45,April 2015,July 2018[582],IPO,70,China,Lei Jun,NaN
4,Alibaba,42,June 2016,September 2014[583],IPO,238,China,Jack Ma,NaN
...,...,...,...,...,...,...,...,...,...
201,Zimi,1+,February 2015[184],March 2021[836],Acquired,0.4,China,NaN,NaN
202,QingCloud,1+,June 2017[184],March 2021[837],IPO,0.46,China,NaN,NaN
203,Novogene,1+,November 2016[184],April 2021,IPO,NaN,China,NaN,NaN
204,MissFresh,1+,December 2017[184],June 2021[838],IPO,2.5,China,NaN,NaN


# gender generation

In [123]:
print(currentUnicorns_df.head())
print(pastUnicorns_df.head())

     Company Valuation(US$ billions)              Valuation date  \
0     SpaceX                    1250  February 2026(2026-02)[20]   
1     OpenAI                     852     March 2026(2026-03)[21]   
2  Anthropic                     380  February 2026(2026-02)[22]   
3  ByteDance                     600              April 2026[23]   
4     Stripe                     159  February 2026(2026-02)[24]   

                            Industry          Country/countries  \
0  Aerospace,Artificial Intelligence              United States   
1            Artificial intelligence              United States   
2            Artificial Intelligence              United States   
3                           Internet                      China   
4                 Financial services  United States and Ireland   

                                          Founder(s)  
0                                          Elon Musk  
1  Sam Altman,Elon Musk,Greg Brockman,Ilya Sutskever  
2  Dario Amodei,Daniela

In [124]:
print("Predicting genders for current unicorns...")
currentUnicorns_df['Founder_Genders'] = filter_df(currentUnicorns_df, "Founder(s)", get_founder_gender)

# Save intermediate result
currentUnicorns_df.to_csv("processed_data/current_unicorns_with_gender.csv", index=False)

df_companies_founders_gender = currentUnicorns_df[['Company', 'Founder_Genders', 'Industry']].copy()
# Note: get_founder_gender now returns normalized 'male', 'female', 'unknown'
print("Done.")

Predicting genders for current unicorns...
Done.


# Tabellen-Daten 1: Gender Pro Industrie

In [137]:
import pandas as pd

# start from currentUnicorns_df which already has Company, Founder_Genders, Industry
df = currentUnicorns_df[['Company', 'Founder_Genders', 'Industry']].copy()

def normalize_founders(g):
    if isinstance(g, (list, tuple)):
        return [str(x).strip().lower() for x in g]
    if pd.isna(g) or g == '':
        return ['unknown']
    return [str(g).strip().lower()]

df['Founder_Genders_list'] = df['Founder_Genders'].apply(normalize_founders)

# normalize Industry into a list
col = 'Industry'
df[col] = df[col].fillna('')
df[col + '_norm'] = (df[col]
                     .str.replace(';', '|', regex=False)
                     .str.replace(',', '|', regex=False)
                     .str.replace('/', '|', regex=False)
                     .str.replace('&', '|', regex=False)
                     .str.replace(' and ', '|', regex=False)
                     .str.replace(' & ', '|', regex=False)
                    )
df[col + '_list'] = df[col + '_norm'].apply(lambda s: [x.strip().title() for x in s.split('|') if x.strip()])

# explode so each row is one (company, item, gender)
df_exp = df.explode(col + '_list').reset_index(drop=True)
df_exp = df_exp[df_exp[col + '_list'] != '']
df_exp = df_exp.explode('Founder_Genders_list').reset_index(drop=True)

# aggregate counts
counts = (df_exp
          .groupby([col + '_list', 'Founder_Genders_list'])
          .size()
          .unstack(fill_value=0))

# ensure columns exist
for gender in ['male', 'female', 'unknown']:
    if gender not in counts.columns:
        counts[gender] = 0

sectors_gender_counts = counts.reset_index().rename(columns={col + '_list': 'Sector'})
sectors_gender_counts = sectors_gender_counts[['Sector', 'male', 'female', 'unknown']]

sectors_gender_counts.to_csv('processed_data/sectors_gender_counts.csv', index=False)
display(sectors_gender_counts.head())


Founder_Genders_list,Sector,male,female,unknown
0,Aerospace,1,0,0
1,Ai,2,0,0
2,Ai Audio Technologies,2,0,0
3,Ai-Poweredsaas,0,1,0
4,Application Security,2,0,0


In [138]:
male_count = 0
female_count = 0
unknown_count = 0

for cell in df_companies_founders_gender['Founder_Genders']:
    if isinstance(cell, (list, tuple)):
        for v in cell:
            if v == 'male':
                male_count += 1
            elif v == 'female':
                female_count += 1
            elif v == 'unknown':
                unknown_count += 1
    else:
        if cell == 'male':
            male_count += 1
        elif cell == 'female':
            female_count += 1
        elif cell == 'unknown':
            unknown_count += 1

print("male:", male_count)
print("female:", female_count)
print("unknown:", unknown_count)


male: 318
female: 28
unknown: 74


In [139]:
unknown_names = []

for i, cell in df_companies_founders_gender['Founder_Genders'].items():
    if isinstance(cell, (list, tuple)):
        for name in cell:
            if name == 'unknown':
                unknown_names.append((i, name))
    else:
        if cell == 'unknown':
            unknown_names.append((i, cell))

# Print rows (index) that have unknowns and the count per row
for idx, _ in unknown_names:
    print(f"Row {idx} contains 'unknown'")

# If you want the total and unique indices:
print("Total unknowns:", len(unknown_names))
print("Rows with unknowns:", sorted({idx for idx, _ in unknown_names}))


Row 3 contains 'unknown'
Row 3 contains 'unknown'
Row 4 contains 'unknown'
Row 6 contains 'unknown'
Row 15 contains 'unknown'
Row 16 contains 'unknown'
Row 17 contains 'unknown'
Row 34 contains 'unknown'
Row 35 contains 'unknown'
Row 43 contains 'unknown'
Row 48 contains 'unknown'
Row 56 contains 'unknown'
Row 57 contains 'unknown'
Row 60 contains 'unknown'
Row 63 contains 'unknown'
Row 71 contains 'unknown'
Row 82 contains 'unknown'
Row 82 contains 'unknown'
Row 88 contains 'unknown'
Row 90 contains 'unknown'
Row 91 contains 'unknown'
Row 101 contains 'unknown'
Row 127 contains 'unknown'
Row 127 contains 'unknown'
Row 127 contains 'unknown'
Row 127 contains 'unknown'
Row 132 contains 'unknown'
Row 141 contains 'unknown'
Row 141 contains 'unknown'
Row 152 contains 'unknown'
Row 152 contains 'unknown'
Row 166 contains 'unknown'
Row 175 contains 'unknown'
Row 175 contains 'unknown'
Row 175 contains 'unknown'
Row 179 contains 'unknown'
Row 181 contains 'unknown'
Row 183 contains 'unknown'

# Tabelle-Daten 2: Land + Geschlecht

In [140]:
import pandas as pd

# start from currentUnicorns_df which already has Company, Founder_Genders, and "Country/countries"
df = currentUnicorns_df[['Company', 'Founder_Genders', 'Country/countries']].copy()

def normalize_founders(g):
    if isinstance(g, (list, tuple)):
        return [str(x).strip().lower() for x in g]
    if pd.isna(g) or g == '':
        return ['unknown']
    return [str(g).strip().lower()]

df['Founder_Genders_list'] = df['Founder_Genders'].apply(normalize_founders)

# normalize Country/countries into a list
col = 'Country/countries'
df[col] = df[col].fillna('')
df[col + '_norm'] = (df[col]
                     .str.replace(';', '|', regex=False)
                     .str.replace(',', '|', regex=False)
                     .str.replace('/', '|', regex=False)
                     .str.replace('&', '|', regex=False)
                     .str.replace(' and ', '|', regex=False)
                     .str.replace(' & ', '|', regex=False)
                    )
df[col + '_list'] = df[col + '_norm'].apply(lambda s: [x.strip().title() for x in s.split('|') if x.strip()])

# explode so each row is one (company, item, gender)
df_exp = df.explode(col + '_list').reset_index(drop=True)
df_exp = df_exp[df_exp[col + '_list'] != '']
df_exp = df_exp.explode('Founder_Genders_list').reset_index(drop=True)

# aggregate counts
counts = (df_exp
          .groupby([col + '_list', 'Founder_Genders_list'])
          .size()
          .unstack(fill_value=0))

# ensure columns exist
for gender in ['male', 'female', 'unknown']:
    if gender not in counts.columns:
        counts[gender] = 0

final_df = counts.reset_index().rename(columns={col + '_list': 'Country'})
final_df = final_df[['Country', 'male', 'female', 'unknown']]

final_df.to_csv('processed_data/countries_gender_counts.csv', index=False)
display(final_df.head())


Founder_Genders_list,Country,male,female,unknown
0,Argentina,1,0,0
1,Armenia,3,0,0
2,Artificial Intelligence,0,0,1
3,Australia,6,1,0
4,Bangladesh,2,0,0


# Tabellendaten 4: Exit Valuation

In [141]:
print("Predicting genders for past unicorns...")
pastUnicorns_df['Founder_Genders'] = filter_df(pastUnicorns_df, "Founders", get_founder_gender)

# Save intermediate result
pastUnicorns_df.to_csv("processed_data/past_unicorns_with_gender.csv", index=False)
print("Done.")

Predicting genders for past unicorns...
Done.


In [142]:
import pandas as pd

# start from currentUnicorns_df which already has Company, Founder_Genders, and "Exit valuation(US$billions)"
df = pastUnicorns_df[['Company', 'Founder_Genders', 'Exit valuation(US$billions)']].copy()

def normalize_founders(g):
    if isinstance(g, (list, tuple)):
        return [str(x).strip().lower() for x in g]
    if pd.isna(g) or g == '':
        return ['unknown']
    return [str(g).strip().lower()]

df['Founder_Genders_list'] = df['Founder_Genders'].apply(normalize_founders)

# normalize Exit valuation(US$billions) into a list
col = 'Exit valuation(US$billions)'
df[col] = df[col].fillna('')
df[col + '_norm'] = (df[col]
                     .str.replace(';', '|', regex=False)
                     .str.replace(',', '|', regex=False)
                     .str.replace('/', '|', regex=False)
                     .str.replace('&', '|', regex=False)
                     .str.replace(' and ', '|', regex=False)
                     .str.replace(' & ', '|', regex=False)
                    )
df[col + '_list'] = df[col + '_norm'].apply(lambda s: [x.strip() for x in s.split('|') if x.strip()])
# explode so each row is one (company, item, gender)
df_exp = df.explode(col + '_list').reset_index(drop=True)
df_exp = df_exp[df_exp[col + '_list'] != '']
df_exp = df_exp.explode('Founder_Genders_list').reset_index(drop=True)

# aggregate counts
counts = (df_exp
          .groupby([col + '_list', 'Founder_Genders_list'])
          .size()
          .unstack(fill_value=0))

# ensure columns exist
for gender in ['male', 'female', 'unknown']:
    if gender not in counts.columns:
        counts[gender] = 0

final_df = counts.reset_index().rename(columns={col + '_list': 'Exit Valuation'})
final_df = final_df[['Exit Valuation', 'male', 'female', 'unknown']]

final_df.to_csv('processed_data/exitval_gender_counts.csv', index=False)
display(final_df.head())


Founder_Genders_list,Exit Valuation,male,female,unknown
0,0.1,2,1,0
1,0.57,0,0,1
2,1,5,0,0
3,1.2,3,0,0
4,1.3,4,0,0


In [143]:
import pandas as pd

# start from currentUnicorns_df which already has Company, Founder_Genders, and "Exit valuation(US$billions)"
df = pastUnicorns_df[['Company', 'Founder_Genders', 'Exit valuation(US$billions)']].copy()

def normalize_founders(g):
    if isinstance(g, (list, tuple)):
        return [str(x).strip().lower() for x in g]
    if pd.isna(g) or g == '':
        return ['unknown']
    return [str(g).strip().lower()]

df['Founder_Genders_list'] = df['Founder_Genders'].apply(normalize_founders)

# normalize Exit valuation(US$billions) into a list
col = 'Exit valuation(US$billions)'
df[col] = df[col].fillna('')
df[col + '_norm'] = (df[col]
                     .str.replace(';', '|', regex=False)
                     .str.replace(',', '|', regex=False)
                     .str.replace('/', '|', regex=False)
                     .str.replace('&', '|', regex=False)
                     .str.replace(' and ', '|', regex=False)
                     .str.replace(' & ', '|', regex=False)
                    )
df[col + '_list'] = df[col + '_norm'].apply(lambda s: [x.strip() for x in s.split('|') if x.strip()])
# explode so each row is one (company, item, gender)
df_exp = df.explode(col + '_list').reset_index(drop=True)
df_exp = df_exp[df_exp[col + '_list'] != '']
df_exp = df_exp.explode('Founder_Genders_list').reset_index(drop=True)

# aggregate counts
counts = (df_exp
          .groupby([col + '_list', 'Founder_Genders_list'])
          .size()
          .unstack(fill_value=0))

# ensure columns exist
for gender in ['male', 'female', 'unknown']:
    if gender not in counts.columns:
        counts[gender] = 0

final_df = counts.reset_index().rename(columns={col + '_list': 'Exit Valuation'})
final_df = final_df[['Exit Valuation', 'male', 'female', 'unknown']]

final_df.to_csv('processed_data/exitval_gender_counts.csv', index=False)
display(final_df.head())


Founder_Genders_list,Exit Valuation,male,female,unknown
0,0.1,2,1,0
1,0.57,0,0,1
2,1,5,0,0
3,1.2,3,0,0
4,1.3,4,0,0


# Planung

Stats:
male: 305
female: 23
unknown: 88

Tabelle 1: Industry + (male, female, unknown)


Tabelle 2: Land + (male, female, unknown)


Tabelle 3: Top 5 Industry in Land + (male, female, unknown)


Tabelle 4: EXIT Wert pro (male, female, unkown)


Visualisierungen

# Rescue Unknowns
Use the new improved web-search logic to fix remaining unknowns without re-processing the whole dataset.

In [144]:
# ---------------------------------------------------------------------------
# RESCUE UNKNOWNS: Targeted re-processing of missing genders
# ---------------------------------------------------------------------------
import pandas as pd
from get_gender_data import get_founder_gender

def rescue_unknowns(df, founder_col, gender_col):
    """Re-process only rows where gender is unknown or contains unknown."""
    mask = df[gender_col].apply(lambda x: 'unknown' in str(x).lower() if x is not None else True)
    unknown_count = mask.sum()
    
    if unknown_count == 0:
        print(f"No unknowns found in {gender_col}.")
        return df
    
    print(f"Attempting to rescue {unknown_count} unknowns in {gender_col}...")
    
    # Targeted update
    df.loc[mask, gender_col] = df.loc[mask, founder_col].apply(lambda x: get_founder_gender(x, use_web_search=True))
    
    new_unknown_count = df[gender_col].apply(lambda x: 'unknown' in str(x).lower()).sum()
    print(f"Resolution complete. Unknowns remaining: {new_unknown_count} (Rescued {unknown_count - new_unknown_count})")
    return df

print("Processing Current Unicorns...")
currentUnicorns_df = rescue_unknowns(currentUnicorns_df, 'Founder(s)', 'Founder_Genders')

print("\nProcessing Past Unicorns...")
pastUnicorns_df = rescue_unknowns(pastUnicorns_df, 'Founders', 'Founder_Genders')

# Save the improved data
currentUnicorns_df.to_csv('processed_data/current_unicorns_rescued.csv', index=False)
pastUnicorns_df.to_csv('processed_data/past_unicorns_rescued.csv', index=False)


Processing Current Unicorns...
Attempting to rescue 54 unknowns in Founder_Genders...


TypeError: get_founder_gender() got an unexpected keyword argument 'use_web_search'